Lead IQ lead scoring model V1 — train + deploy

Built from the trimmed `training.ipynb`, with two additions:
1. Reads training data from `LEADIQ_TRAINING_SET_V6_FS` — the Feature-Store-enriched table `fs_code.ipynb` materializes — instead of the hand-written SQL join.
2. Saves the trained model to a dedicated `LEADIQ_MODEL_STAGE`, keeping every run's snapshot (timestamped) plus a `latest` pointer.

Run `fs_code.ipynb` first -- this notebook's first cell depends on `LEADIQ_TRAINING_SET_V6_FS` existing. NOT YET RUN / VERIFIED end-to-end; same caveat as `fs_code.ipynb` -- read-only SQL is all I could check this against.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql("USE ROLE DATA_SCIENTIST").collect()

# Separate from the existing shared ML_OPS.MODEL_STAGE -- Lead IQ gets its own,
# per the ask. Idempotent: no-op if it already exists.
session.sql("""
    CREATE STAGE IF NOT EXISTS AWS_DP_PROD.ML_OPS.LEADIQ_MODEL_STAGE
    DIRECTORY = (ENABLE = TRUE)
    COMMENT = 'Lead IQ model artifacts -- one timestamped snapshot per training run, plus a latest pointer'
""").collect()
print('stage ready: AWS_DP_PROD.ML_OPS.LEADIQ_MODEL_STAGE')

In [ ]:
import pandas as pd, numpy as np

# read straight from the Feature-Store-enriched table (fs_code.ipynb),
# which already carries the correct person-level LABEL plus all 17 features
# (no GA4, no Category Prior -- dropped from the feature store per instruction).
df = session.table("AWS_DP_PROD.ML_OPS.LEADIQ_TRAINING_SET_V6_FS").to_pandas()

drop_cols = ['LEAD_ID', 'LEAD_TS', 'LABEL']
features  = [c for c in df.columns if c not in drop_cols]

df['LEAD_TS'] = pd.to_datetime(df['LEAD_TS'], utc=True).dt.tz_localize(None)
train_end, test_start = pd.Timestamp('2024-04-30'), pd.Timestamp('2024-08-28')
test_end = df['LEAD_TS'].max() - pd.DateOffset(months=15)

tr = df[df['LEAD_TS'] <= train_end]
te = df[(df['LEAD_TS'] >= test_start) & (df['LEAD_TS'] <= test_end)].reset_index(drop=True)
X_tr, y_tr = tr[features], tr['LABEL'].values
X_te, y_te = te[features], te['LABEL'].values
spw = (y_tr==0).sum() / (y_tr==1).sum()
print(f'train {len(tr):,}  test {len(te):,}  test_pos {y_te.mean():.4f}  spw {spw:.0f}')

In [ ]:
from xgboost import XGBClassifier
import joblib
from datetime import datetime, timezone

params = dict(n_estimators=300, max_depth=6, learning_rate=0.1,
              scale_pos_weight=spw, eval_metric='aucpr',
              missing=np.nan, random_state=42, n_jobs=-1)
model = XGBClassifier(**params).fit(X_tr, y_tr)
p_te = model.predict_proba(X_te)[:, 1]

# Snowflake notebook CWD is read-only -- every local file must be written
# under /tmp/, not a bare filename in the working directory.
#
# joblib, not XGBoost's native save_model/JSON -- matching the recommendation
# system's convention (joblib.dump / joblib.load) rather than XGBoost's own
# recommended native format. Trade-off accepted deliberately: joblib pickles
# the whole fitted sklearn wrapper (one call to load, no reconstruction
# needed), but is less portable across xgboost/sklearn/numpy version changes
# than the native JSON/UBJ format -- kept for consistency with how the team
# already loads models, not because it's the more robust choice on its own.
ts = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
snapshot_name = f'leadiq_persontarget_v1_{ts}.joblib'
latest_name = 'leadiq_persontarget_v1_latest.joblib'
snapshot_path = f'/tmp/{snapshot_name}'
latest_path   = f'/tmp/{latest_name}'

joblib.dump(model, snapshot_path)
joblib.dump(model, latest_path)

stage = '@AWS_DP_PROD.ML_OPS.LEADIQ_MODEL_STAGE'

def put_and_check(local_path, stage, **kwargs):
    """session.file.put doesn't always raise on failure -- it can return a
    result row with status ERROR/SKIPPED instead. Check it explicitly rather
    than assume the call succeeded just because it didn't throw."""
    results = session.file.put(local_path, stage, **kwargs)
    for r in results:
        status = getattr(r, 'status', None) or r['status']
        if status not in ('UPLOADED', 'SKIPPED'):
            raise RuntimeError(f'upload failed for {local_path}: {r}')
        print(f'  {status}: {local_path} -> {stage}  ({r})')
    return results

put_and_check(snapshot_path, stage, auto_compress=False, overwrite=False)
put_and_check(latest_path,   stage, auto_compress=False, overwrite=True)

print(f'snapshot: {stage}/{snapshot_name}')
print(f'latest:   {stage}/{latest_name}  (overwritten)')

# A single new lead's raw XGBoost probability has no meaning as a 1-100 score
# by itself -- score_1_100 below is that raw score's PERCENTILE within a
# reference population (this held-out test set). predict() at real inference
# time only ever sees one lead, so it needs this population frozen and saved
# now, matched to this exact model snapshot -- not recomputed from a
# population it will never have.
ref_scores_name   = f'leadiq_persontarget_v1_{ts}_refscores.npy'
ref_scores_latest = 'leadiq_persontarget_v1_latest_refscores.npy'
ref_scores_path        = f'/tmp/{ref_scores_name}'
ref_scores_latest_path = f'/tmp/{ref_scores_latest}'
np.save(ref_scores_path, np.sort(p_te))
np.save(ref_scores_latest_path, np.sort(p_te))
put_and_check(ref_scores_path,        stage, auto_compress=False, overwrite=False)
put_and_check(ref_scores_latest_path, stage, auto_compress=False, overwrite=True)
print(f'reference scores: {stage}/{ref_scores_name}  (n={len(p_te):,})')

In [ ]:
# Same 3 checks evaluation.ipynb / model_eval.ipynb already computed --
# now sourced from model_eval.py instead of duplicated inline, matching
# how the recommendation system's training notebook calls model_eval
# right after training. y_te/p_te are already in scope from the cells
# above.
import model_eval

metrics = model_eval.lift_auc_metrics(y_te, p_te)
dl = model_eval.eval_plots_and_decile_lift(y_te, p_te)
hot_stats = model_eval.hot_threshold_confusion_matrix(y_te, p_te)

In [ ]:
# Only needed to make the Database Explorer / Horizon Catalog file
# browser show the new uploads immediately -- that panel reads a
# separate, slower directory-metadata layer than LIST/file.get do.
# Nothing else in this notebook or model_inference.py depends on this;
# session.file.get always reads current data with no refresh needed.
session.sql("ALTER STAGE AWS_DP_PROD.ML_OPS.LEADIQ_MODEL_STAGE REFRESH").collect()

In [ ]:
imp = model_eval.feature_importance_plot(model, features)

In [ ]:
src_cols = [c for c in features if c.startswith('SRC_')]
seg_table = model_eval.segment_lift_table(te, p_te, src_cols)

In [ ]:
# Map score -> 1-100 percentile, then -> PRD tier (Hot 75-100 / Warm 40-74 / Cold 1-39)
scored = model_eval.build_scored_output(te, p_te)
print(scored['tier'].value_counts())
scored.head(10)

In [ ]:
scored_ts_table = model_eval.write_scored_table(session, scored, ts)

In [ ]:
pip install mlflow

In [ ]:
%%sql -r dataframe_1
USE ROLE DATA_SCIENTIST

In [ ]:
# Mirrors the recommendation system's tracking setup exactly -- same
# shared MLflow server, a NEW experiment for Lead IQ rather than
# reusing RECOS's. Needs AWS_DP_PROD.ML_OPS.MLFLOW_CREDS mounted on
# this notebook's service -- confirm that's attached before running
# this cell; it isn't guaranteed to be, the same way LEADIQ_PYPI_ACCESS
# wasn't attached until it was explicitly requested earlier.
!pip install mlflow
import os
import mlflow
_sec = os.environ["SNOWFLAKE_CONTAINER_SERVICES_SECRET_PATH_PREFIX"]
_base = f"{_sec}/aws_dp_prod/ml_ops/mlflow_creds"

os.environ["MLFLOW_TRACKING_URI"] = "http://prod-m-mlflo-ibfhggxtqvwe-1711046187.us-east-1.elb.amazonaws.com"
os.environ["MLFLOW_TRACKING_USERNAME"] = open(f"{_base}/username").read().strip()
os.environ["MLFLOW_TRACKING_PASSWORD"] = open(f"{_base}/password").read().strip()

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("leadiq_scoringservice")
print('tracking to leadiq_scoringservice')

In [ ]:
# Logs THIS run's exact snapshot (snapshot_path, not latest_path) --
# what gets deployed later should be traceable to one immutable
# training run, not a 'latest' pointer that could move under it.
#
# connection_params is metadata only (no secret in it) -- matches
# what's confirmed from the recommendation system's real dict.
# LEADIQ_ML_INFERENCE_USER doesn't exist yet (request still pending) --
# logging succeeds regardless, since nothing here is validated against
# a real identity until something actually tries to LOAD this model.
import joblib
from datetime import datetime, timezone
latest_name = 'leadiq_persontarget_v1_latest.joblib'
latest_path   = f'/tmp/{latest_name}'


connection_params = {
    "account": "BOATSGROUP-DATA",
    "user": "LEADIQ_ML_INFERENCE_USER",
    "role": "ML_INFERENCE_ROLE",
    "warehouse": "COMPUTE_WH",
    "database": "AWS_DP_PROD",
    "schema": "BOATS_FEATURE_STORE",
    "online_store_query_url": "https://bzh467hd-boatsgroup-data.snowflakecomputing.app",
}

artifacts = {
    "leadiq_model": snapshot_path,       # THIS run's .joblib, from train_model_and_snapshot
    "leadiq_refscores": ref_scores_path,  # THIS run's matching refscores.npy
}

# No shap/numba/llvmlite chain -- Lead IQ doesn't use SHAP, unlike the
# recommendation system, so none of that pin chain applies here.
pip_requirements = [
    "mlflow==3.12.0",
    "joblib",
    "pandas",
    "cryptography",
    # MUST stay exact -- a floating >=1.36 re-resolving on a container
    # rebuild is exactly what crash-looped the recommendation system's
    # own deployment once already.
    "snowflake-ml-python==1.42.0",
    "snowflake-connector-python[pandas]",
    "snowflake-snowpark-python",
    # xgboost/scikit-learn intentionally left unpinned -- snowflake-ml-python
    # bounds them already and both ship reliable wheels (same reasoning
    # the recommendation system's own pip_requirements uses).
]

import model_inference
code_paths = [model_inference.__file__]
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
with mlflow.start_run(run_name=f"run_{timestamp}") as run:
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=model_inference.__file__,
        artifacts=artifacts,
        code_paths=code_paths,
        pip_requirements=pip_requirements,
        model_config={"model_version": "latest"},
    )
    print(f"Model logged successfully in run: {run.info.run_id}")